In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import matplotlib.pyplot as plt

In [38]:
df_train = pd.read_csv("../data/processed/train_data.csv")
df_test = pd.read_csv("../data/processed/test_data.csv")

In [39]:
df_train["hasMonthlyIncome"] = df_train["MonthlyIncome"].apply(lambda x: 0 if np.isnan(x) else 1)
df_train["hasNumberOfDependents"] = df_train["NumberOfDependents"].apply(lambda x: 0 if np.isnan(x) else 1)
df_test["hasMonthlyIncome"] = df_test["MonthlyIncome"].apply(lambda x: 0 if np.isnan(x) else 1)
df_test["hasNumberOfDependents"] = df_test["NumberOfDependents"].apply(lambda x: 0 if np.isnan(x) else 1)

In [40]:
df_0 = df_train[~df_train["MonthlyIncome"].isna()].reset_index(drop=True)
df_1 = df_train[~df_train["NumberOfDependents"].isna()].reset_index(drop=True)

df_use_0 = df_0[[ 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       "hasMonthlyIncome","hasNumberOfDependents"]]

df_use_1 = df_1[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       "hasMonthlyIncome","hasNumberOfDependents"]]


target_income = df_0['MonthlyIncome']
target_dependents = df_1['NumberOfDependents']



In [ ]:
model_income = RandomForestRegressor()
model_income.fit(df_use_0, target_income)

model_dependents = RandomForestRegressor()
model_dependents.fit(df_use_1, target_dependents)

In [41]:
def retornaValorIncome(x):
    X = x[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       "hasMonthlyIncome","hasNumberOfDependents"]]
    return model_income.predict(X)

def retornaValorDependents(x):
    X = x[['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       "hasMonthlyIncome","hasNumberOfDependents"]]
    return model_dependents.predict(X)

In [42]:
mask_income = df_train["MonthlyIncome"].isnull()
df_train.loc[mask_income, "MonthlyIncome"] = retornaValorIncome(df_train[mask_income])

mask_dep = df_train["NumberOfDependents"].isnull()
df_train.loc[mask_dep, "NumberOfDependents"] = retornaValorDependents(df_train[mask_dep])

mask_income = df_test["MonthlyIncome"].isnull()
df_test.loc[mask_income, "MonthlyIncome"] = retornaValorIncome(df_test[mask_income])

mask_dep = df_test["NumberOfDependents"].isnull()
df_test.loc[mask_dep, "NumberOfDependents"] = retornaValorDependents(df_test[mask_dep])

In [43]:
column_mapping = {'SeriousDlqin2yrs': 'Target', 'RevolvingUtilizationOfUnsecuredLines': 'credit_utilization', 'age': 'age', 
                  'NumberOfTime30-59DaysPastDueNotWorse': 'late_30_59_days', 'DebtRatio': 'debt_ratio', 'MonthlyIncome': 'monthly_income', 
                  'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines', 'NumberOfTimes90DaysLate': 'late_90_days', 
                  'NumberRealEstateLoansOrLines': 'real_estate_loans', 'NumberOfTime60-89DaysPastDueNotWorse': 'late_60_89_days', 
                  'NumberOfDependents': 'dependents', "hasMonthlyIncome" :"hasIncome" ,"hasNumberOfDependents": "hasDependents" }

df_train = df_train.rename(columns=column_mapping)

In [44]:
cols_clip = ['credit_utilization','late_30_59_days', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'late_90_days','real_estate_loans', 'late_60_89_days', 'dependents']

In [45]:
for col in cols_clip:
    lower = df_train[col].quantile(0.01)
    upper = df_train[col].quantile(0.99)
    df_train[col] = df_train[col].clip(lower=lower, upper=upper)

In [46]:
df_train["income_per_dependent"] = df_train["monthly_income"] / (df_train["dependents"] + 1)
df_train["weighted_late"] = ((1 * df_train['late_30_59_days']) + (2 * df_train['late_60_89_days']) + (3 * df_train['late_90_days']))

In [47]:
df_X = df_train.drop("Target",axis=1)
df_y = df_train[["Target"]]

In [48]:
from sklearn.model_selection import train_test_split

In [50]:
X_train.to_parquet("../data/processed/X_train.parquet") 
X_test.to_parquet("../data/processed/X_test.parquet")
y_train.to_parquet("../data/processed/y_train.parquet")
y_test.to_parquet("../data/processed/y_test.parquet") 